# Exploración del Modelo 3: múltiples edificios y costo por tipo de transición

En este notebook se documenta la implementación y validación del Modelo 3 del problema de asignación de salas con movilidad entre bloques consecutivos.

El Modelo 3 extiende el Modelo 2 incorporando una estructura de campus con múltiples edificios. La diferencia central está en la matriz de costos:

- si un estudiante permanece en la **misma sala**, el costo es 0;
- si cambia de sala pero **dentro del mismo edificio**, el costo es 1;
- si cambia a una sala de **otro edificio**, el costo es $\gamma_{e_1, e_2}$, donde $\gamma$ es una matriz de penalización entre edificios.

Se mantienen las capacidades heterogéneas de salas y cursos del Modelo 2, así como la posibilidad de que estudiantes queden libres entre bloques.

El propósito de este notebook es mostrar cómo se generaron los datos sintéticos, cómo se validaron, cómo se resolvió el modelo con Gurobi y cómo se compararon los resultados obtenidos con los resultados esperados.

## Objetivo del notebook

Este notebook tiene como objetivo principal verificar que la implementación computacional del Modelo 3 funciona correctamente sobre una batería de tests sintéticos.

En particular, se busca revisar que:

1. Los datos sintéticos estén bien construidos, incluyendo la asignación de salas a edificios.
2. Se cumpla la conservación de estudiantes entre bloques.
3. Las capacidades de las salas sean consideradas correctamente.
4. La matriz de costos refleje correctamente los tres tipos de transición.
5. El modelo detecte casos infactibles cuando corresponde.
6. La función objetivo entregue el valor esperado.
7. La implementación en Gurobi respete todas las restricciones del modelo.

## Descripción del Modelo 3

El Modelo 3 corresponde a una versión del problema con múltiples edificios y costo dependiente del tipo de transición entre salas.

Los principales conjuntos del modelo son:

- $I$: conjunto de cursos del bloque 1.
- $K$: conjunto de cursos del bloque 2.
- $R$: conjunto de salas disponibles.
- $E$: conjunto de edificios del campus.

Los principales parámetros son:

- $n_i$: tamaño del curso $i$ del bloque 1.
- $m_k$: tamaño del curso $k$ del bloque 2.
- $cap_r$: capacidad de la sala $r$.
- $ed_r \in E$: edificio al que pertenece la sala $r$.
- $f_{ik}$: cantidad de estudiantes que pasan desde el curso $i$ al curso $k$.
- $f_{iL}$: cantidad de estudiantes del curso $i$ que quedan libres en el segundo bloque.
- $\gamma_{e_1, e_2}$: penalización por cambiar del edificio $e_1$ al edificio $e_2$ (con $\gamma_{e,e} = 1$).
- $c_{rs}$: costo de moverse desde la sala $r$ hacia la sala $s$.

En esta versión, el costo entre salas depende del tipo de transición:

$$
c_{rs} =
\begin{cases}
0, & \text{si } r = s, \\
1, & \text{si } r \neq s \text{ y } ed_r = ed_s, \\
\gamma_{ed_r, ed_s}, & \text{si } ed_r \neq ed_s.
\end{cases}
$$

La función objetivo sigue siendo minimizar la movilidad total ponderada de estudiantes entre bloques:

$$
\min \sum_{i \in I} \sum_{k \in K} \sum_{r \in R} \sum_{s \in R} f_{ik} \cdot c_{rs} \cdot z_{ikrs}
$$

## Restricciones principales del Modelo 3

El modelo hereda todas las restricciones del Modelo 2 y no agrega restricciones nuevas. La extensión está enteramente en la función de costo.

### 1. Asignación única

Cada curso del bloque 1 debe ser asignado exactamente a una sala:

$$\sum_{r \in R} x_{ir} = 1 \quad \forall i \in I$$

Cada curso del bloque 2 también debe ser asignado exactamente a una sala:

$$\sum_{s \in R} y_{ks} = 1 \quad \forall k \in K$$

### 2. Ocupación única

Cada sala puede recibir a lo más un curso en cada bloque:

$$\sum_{i \in I} x_{ir} \leq 1 \quad \forall r \in R$$

$$\sum_{k \in K} y_{ks} \leq 1 \quad \forall s \in R$$

### 3. Capacidad suficiente

Si un curso se asigna a una sala, debe caber en ella:

$$n_i x_{ir} \leq cap_r \quad \forall i \in I, \forall r \in R$$

$$m_k y_{ks} \leq cap_s \quad \forall k \in K, \forall s \in R$$

### 4. Linealización

La variable auxiliar $z_{ikrs}$ representa que el curso $i$ está en la sala $r$ **y** el curso $k$ está en la sala $s$:

$$z_{ikrs} \leq x_{ir}$$

$$z_{ikrs} \leq y_{ks}$$

$$z_{ikrs} \geq x_{ir} + y_{ks} - 1$$

## Estructura de archivos usada

Para mantener el proyecto ordenado, se separó el trabajo en distintos archivos `.py`.

La estructura usada es:

```text
src/
├── generar_data_modelo_3.py
├── validar_data.py
├── modelo_3_gurobi.py
└── correr_tests_modelo_3.py
```

Cada archivo tiene una responsabilidad distinta:

* `generar_data_modelo_3.py`: genera las instancias sintéticas del Modelo 3, incluyendo la asignación de salas a edificios y la matriz de penalización $\gamma$.
* `validar_data.py`: revisa que los datos estén bien construidos (reutiliza la validación del Modelo 2 y agrega chequeos sobre edificios).
* `modelo_3_gurobi.py`: implementa y resuelve el modelo usando Gurobi, con la nueva función de costo.
* `correr_tests_modelo_3.py`: ejecuta todos los tests y compara resultados esperados con obtenidos.

Los datos generados se guardan en:
```text
data/modelo3/
```

## Archivos de cada test

Cada carpeta de test contiene los siguientes archivos:

```text
cursos_b1.csv
cursos_b2.csv
salas.csv          # ahora incluye columna 'edificio'
flujos.csv
libres.csv
costos.csv         # matriz de costos con los 3 tipos de transición
gamma.csv          # penalización entre pares de edificios
metadata.json
```

La novedad respecto al Modelo 2 es que `salas.csv` incluye una columna `edificio` y se agrega el archivo `gamma.csv` con la matriz de penalización entre edificios.

## Descripción de los tests sintéticos

Para validar el Modelo 3 se construyó una batería de tests sintéticos dividida en dos grupos: tests pequeños y tests medianos.

Los **tests pequeños** permiten revisar partes específicas del modelo de forma controlada. Son útiles para detectar errores en la matriz de costos y verificar que el modelo distingue correctamente los tres tipos de transición.

Los **tests medianos** tienen entre 5 y 8 cursos y múltiples edificios. Estos permiten mostrar que el modelo funciona en instancias más representativas del problema real.

| Grupo | Test | Propósito | Resultado esperado |
|---|---|---|---|
| Pequeño | M3_T01_misma_sala_costo_0 | Verificar que permanecer en la misma sala tiene costo 0 | OPTIMAL, costo 0 |
| Pequeño | M3_T02_mismo_edificio_costo_1 | Verificar que cambiar de sala dentro del mismo edificio tiene costo 1 | OPTIMAL, costo > 0 |
| Pequeño | M3_T03_distinto_edificio_costo_gamma | Verificar que cambiar de edificio aplica el costo gamma correspondiente | OPTIMAL, costo con gamma |
| Pequeño | M3_T04_infactible_capacidad | Verificar que el modelo detecta infactibilidad por capacidad | INFEASIBLE |
| Pequeño | M3_T05_estudiantes_libres | Verificar que los estudiantes libres no generan costo | OPTIMAL, costo 0 |
| Mediano | M3_T06_5_cursos_2_edificios | Validar instancia con 5 cursos en 2 edificios | OPTIMAL, costo esperado |
| Mediano | M3_T07_6_cursos_3_edificios_con_libres | Validar instancia con 6 cursos en 3 edificios y estudiantes libres | OPTIMAL, costo 0 |
| Mediano | M3_T08_8_cursos_edificio_cambia_solucion | Verificar que los edificios pueden cambiar la solución óptima | OPTIMAL, costo esperado |
| Mediano | M3_T09_8_cursos_multiples_optimos | Validar instancia con múltiples soluciones óptimas en distintos edificios | OPTIMAL, costo esperado |

Esta estructura permite validar el modelo de forma progresiva: primero se comprueban los tres tipos de transición por separado, luego se verifica la detección de infactibilidad y el manejo de libres, y finalmente se resuelven instancias medianas con múltiples edificios.

## Matriz de costos del Modelo 3

La novedad central del Modelo 3 respecto al Modelo 2 está en la matriz de costos. En el Modelo 2, cualquier cambio de sala tenía costo 1. En el Modelo 3, el costo depende de si las salas pertenecen al mismo edificio o no.

La regla es:

$$
c_{rs} =
\begin{cases}
0, & \text{si } r = s, \\
1, & \text{si } r \neq s \text{ y } ed_r = ed_s, \\
\gamma_{ed_r, ed_s}, & \text{si } ed_r \neq ed_s.
\end{cases}
$$

Por ejemplo, si se tienen 4 salas distribuidas en 2 edificios:

- $r1, r2 \in$ Edificio A
- $r3, r4 \in$ Edificio B
- $\gamma_{A,B} = \gamma_{B,A} = 3$

La matriz de costos resultante es:

| | r1 | r2 | r3 | r4 |
|---|---:|---:|---:|---:|
| **r1** | 0 | 1 | 3 | 3 |
| **r2** | 1 | 0 | 3 | 3 |
| **r3** | 3 | 3 | 0 | 1 |
| **r4** | 3 | 3 | 1 | 0 |

Esta matriz se construye automáticamente en `generar_data_modelo_3.py` a partir del archivo `salas.csv` (que incluye el edificio de cada sala) y del archivo `gamma.csv` (que contiene la penalización entre pares de edificios).

## 1. Preparación del notebook

Primero importamos las librerías necesarias y conectamos el notebook con los archivos `.py` ubicados en la carpeta `src`.

Este notebook usará funciones ya implementadas en:

- `generar_data_modelo_3.py`
- `validar_data.py`
- `modelo_3_gurobi.py`

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

# Detectar la carpeta principal del proyecto
ROOT = Path.cwd()

# Si estamos dentro de la carpeta notebooks, subimos un nivel
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
DATA_MODELO_3 = ROOT / "data" / "modelo3"

# Agregar src al path para poder importar nuestros archivos .py
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("Carpeta principal:", ROOT)
print("Carpeta src:", SRC)
print("Carpeta data/modelo3:", DATA_MODELO_3)

Carpeta principal: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II
Carpeta src: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\src
Carpeta data/modelo3: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo3


In [6]:
from validar_data_3 import validar_instancia_modelo_3
from modelo_3_gurobi import resolver_modelo_3
from generar_data_modelo_3 import generar_tests_pequenos
from generar_data_modelo_3 import generar_tests_medianos
from generar_data_modelo_3 import generar_todos_los_tests

print("Funciones importadas correctamente.")

Funciones importadas correctamente.


## 2. Generación de datos sintéticos

Los datos sintéticos del Modelo 3 se generan con el archivo:

```text
src/generar_data_modelo_3.py
```

La diferencia respecto al Modelo 2 es que ahora cada sala debe tener asignado un edificio, y se requiere especificar la matriz de penalización $\gamma$ entre edificios.

In [7]:
generar_tests_pequenos()

Test generado: M3_T01_misma_sala_costo_0
Test generado: M3_T02_mismo_edificio_costo_1
Test generado: M3_T03_distinto_edificio_costo_gamma
Test generado: M3_T04_infactible_capacidad
Test generado: M3_T05_estudiantes_libres

Tests pequeños del Modelo 3 generados correctamente.


## 3. Tests disponibles

A continuación se revisan las carpetas de tests que existen dentro de `data/modelo3`.

In [8]:
carpetas_tests_pequenos = sorted([
    carpeta for carpeta in DATA_MODELO_3.iterdir()
    if carpeta.is_dir() and carpeta.name.startswith(("M3_T01", "M3_T02", "M3_T03", "M3_T04", "M3_T05"))
])

for carpeta in carpetas_tests_pequenos:
    print(carpeta.name)

M3_T01_misma_sala_costo_0
M3_T02_mismo_edificio_costo_1
M3_T03_distinto_edificio_costo_gamma
M3_T04_infactible_capacidad
M3_T05_estudiantes_libres


## 4. Tabla resumen de los tests

Cada test tiene un archivo `metadata.json` con su descripción, status esperado y objetivo esperado.

In [9]:
resumen_tests = []

for carpeta in carpetas_tests_pequenos:
    ruta_metadata = carpeta / "metadata.json"

    with open(ruta_metadata, "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)

    resumen_tests.append({
        "test": carpeta.name,
        "descripcion": metadata["descripcion"],
        "status_esperado": metadata["expected_status"],
        "objetivo_esperado": metadata["expected_obj"]
    })

tabla_tests = pd.DataFrame(resumen_tests)
tabla_tests

,test,descripcion,status_esperado,objetivo_esperado
0,M3_T01_misma_sala_costo_0,Todos los estudiantes permanecen en la misma s...,OPTIMAL,0.0
1,M3_T02_mismo_edificio_costo_1,Flujos cruzados dentro del mismo edificio. Cos...,OPTIMAL,60.0
2,M3_T03_distinto_edificio_costo_gamma,Parte del flujo cruza de edificio A a B. Costo...,OPTIMAL,30.0
3,M3_T04_infactible_capacidad,Curso i1 tiene tamaño 60 pero la sala más gran...,INFEASIBLE,NaN
4,M3_T05_estudiantes_libres,Estudiantes libres no generan costo. Los que c...,OPTIMAL,0.0


## 5. Inspección de un test pequeño específico

Ahora se revisa con más detalle una instancia particular.

Se analizará el test `M3_T03_distinto_edificio_costo_gamma`, porque permite observar el efecto de la penalización entre edificios y es la extensión más relevante respecto al Modelo 2.

In [10]:
nombre_test = "M3_T03_distinto_edificio_costo_gamma"
carpeta_test = DATA_MODELO_3 / nombre_test

cursos_b1 = pd.read_csv(carpeta_test / "cursos_b1.csv")
cursos_b2 = pd.read_csv(carpeta_test / "cursos_b2.csv")
salas = pd.read_csv(carpeta_test / "salas.csv")
flujos = pd.read_csv(carpeta_test / "flujos.csv")
libres = pd.read_csv(carpeta_test / "libres.csv")
costos = pd.read_csv(carpeta_test / "costos.csv")
gamma = pd.read_csv(carpeta_test / "gamma.csv")

print("Test seleccionado:", nombre_test)

Test seleccionado: M3_T03_distinto_edificio_costo_gamma


### Cursos del bloque 1

In [11]:
cursos_b1

,curso_id,tamano
0,i1,20
1,i2,20


### Cursos del bloque 2

In [12]:
cursos_b2

,curso_id,tamano
0,k1,30


### Salas disponibles

A diferencia del Modelo 2, ahora `salas.csv` incluye la columna `edificio`.

In [13]:
salas

,sala_id,capacidad,edificio
0,r1,20,A
1,r2,20,A
2,r3,30,B


### Matriz de penalización entre edificios ($\gamma$)

In [14]:
gamma

,edificio_origen,edificio_destino,gamma
0,A,A,1
1,A,B,3
2,B,A,3
3,B,B,1


### Matriz de costos entre salas

A diferencia del Modelo 2, ahora `costos.csv` puede tener valores distintos de 0 y 1: los pares de salas de edificios distintos tendrán el costo $\gamma$ correspondiente.

In [15]:
costos.set_index(costos.columns[0])

,sala_destino,costo
sala_origen,,
r1,r1,0
r1,r2,1
r1,r3,3
r2,r1,1
r2,r2,0
r2,r3,3
r3,r1,3
r3,r2,3
r3,r3,0


### Matriz de flujos

El archivo `flujos.csv` está en formato largo. Para entenderlo mejor, se transforma a formato matriz.

In [16]:
F = flujos.pivot(
    index="curso_b1",
    columns="curso_b2",
    values="flujo"
).fillna(0)

F

curso_b2,k1
curso_b1,
i1,20
i2,10


## 6. Validación de datos

Antes de resolver el modelo con Gurobi, se valida que la instancia esté bien construida. Para el Modelo 3 se agregan chequeos adicionales:

- que todas las salas tengan edificio asignado;
- que el archivo `gamma.csv` esté completo para todos los pares de edificios;
- que los valores de $\gamma$ sean positivos.

In [17]:
datos_validados = validar_instancia_modelo_3(carpeta_test)

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 1
  Salas: 3
  Edificios: ['A', 'B']
  Total estudiantes bloque 1: 40
  Total estudiantes bloque 2: 30
  Total estudiantes libres: 10
  Flujo total entre cursos: 30


## 7. Resolución del test con Gurobi

Ahora se resuelve el test seleccionado usando la función `resolver_modelo_3`.

Se usa `mostrar_output=False` para ocultar el log largo de Gurobi.

In [18]:
resultado = resolver_modelo_3(carpeta_test, mostrar_output=False)
resultado

Validación completada correctamente.

Resumen:
  Cursos bloque 1: 2
  Cursos bloque 2: 1
  Salas: 3
  Edificios: ['A', 'B']
  Total estudiantes bloque 1: 40
  Total estudiantes bloque 2: 30
  Total estudiantes libres: 10
  Flujo total entre cursos: 30
Set parameter Username
Set parameter LicenseID to value 2808412
Academic license - for non-commercial use only - expires 2027-04-16


{'status': 'OPTIMAL',
 'objetivo': 30.0,
 'asignacion_b1': {'i1': 'r3', 'i2': 'r1'},
 'asignacion_b2': {'k1': 'r3'}}

### Asignación obtenida para el bloque 1

La siguiente tabla muestra en qué sala queda asignado cada curso del bloque 1.

In [19]:
pd.DataFrame(
    list(resultado["asignacion_b1"].items()),
    columns=["curso_b1", "sala_asignada"]
)

,curso_b1,sala_asignada
0,i1,r3
1,i2,r1


### Asignación obtenida para el bloque 2

La siguiente tabla muestra en qué sala queda asignado cada curso del bloque 2.

In [20]:
pd.DataFrame(
    list(resultado["asignacion_b2"].items()),
    columns=["curso_b2", "sala_asignada"]
)

,curso_b2,sala_asignada
0,k1,r3


## 8. Interpretación del test seleccionado

En el test `M3_T03_distinto_edificio_costo_gamma`, los estudiantes deben cambiar de edificio entre los dos bloques. El modelo aplica correctamente el costo $\gamma$ a estos cambios, lo que hace que el costo total sea mayor que si las salas estuvieran en el mismo edificio.

Este test permite observar directamente que el modelo distingue entre:

- permanecer en la misma sala (costo 0),
- cambiar de sala en el mismo edificio (costo 1),
- cambiar a una sala en otro edificio (costo $\gamma > 1$).

El resultado esperado para este test corresponde al costo total calculado usando el valor de $\gamma$ definido entre los edificios involucrados.

## 9. Tests medianos del Modelo 3

Después de validar los tests pequeños, se agregan tests medianos con entre 5 y 8 cursos y múltiples edificios.

Estos tests buscan mostrar que el modelo no solo funciona en instancias mínimas, sino también en casos más representativos del campus real. Para cada instancia se muestra una breve explicación del resultado esperado y luego se presenta el resultado obtenido.

In [21]:
generar_tests_medianos()

Test generado: M3_T06_5_cursos_2_edificios_gamma_alto
Test generado: M3_T07_6_cursos_3_edificios_con_libres
Test generado: M3_T08_5_cursos_2_edificios_gamma_moderado
Test generado: M3_T09_6_cursos_3_edificios_multiples_optimos

Tests medianos del Modelo 3 generados correctamente.


In [22]:
carpetas_tests_medianos = sorted([
    carpeta for carpeta in DATA_MODELO_3.iterdir()
    if carpeta.is_dir() and carpeta.name.startswith(("M3_T06", "M3_T07", "M3_T08", "M3_T09"))
])

In [23]:
resumen_tests_medianos = []

for carpeta in carpetas_tests_medianos:
    ruta_metadata = carpeta / "metadata.json"

    with open(ruta_metadata, "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)

    resumen_tests_medianos.append({
        "test": carpeta.name,
        "descripcion": metadata["descripcion"],
        "status_esperado": metadata["expected_status"],
        "objetivo_esperado": metadata["expected_obj"]
    })

tabla_tests_medianos = pd.DataFrame(resumen_tests_medianos)
tabla_tests_medianos

,test,descripcion,status_esperado,objetivo_esperado
0,M3_T06_4_cursos_2_edificios_gamma_alto,4 cursos en 2 edificios. gamma=10 entre edific...,OPTIMAL,20
1,M3_T06_5_cursos_2_edificios_gamma_alto,5 cursos en 2 edificios. gamma=8 evita cruces....,OPTIMAL,20
2,M3_T07_4_cursos_2_edificios_con_libres,4 cursos en 2 edificios con libres. Todos perm...,OPTIMAL,0
3,M3_T07_6_cursos_3_edificios_con_libres,6 cursos en 3 edificios con libres. Todos perm...,OPTIMAL,0
4,M3_T08_4_cursos_edificio_cambia_solucion,Flujos cruzados con 4 salas en 2 edificios y g...,OPTIMAL,20
5,M3_T08_5_cursos_2_edificios_gamma_moderado,5 cursos en 2 edificios. gamma=2. Capacidades ...,OPTIMAL,20
6,M3_T09_4_cursos_4_edificios_cruce_forzado,4 cursos en 4 edificios distintos. Flujos dire...,OPTIMAL,0
7,M3_T09_6_cursos_3_edificios_multiples_optimos,6 cursos en 3 edificios con flujos simétricos....,OPTIMAL,102


In [24]:
from IPython.display import display

def mostrar_resultado_y_asignacion(nombre_test):
    """
    Resuelve un test con Gurobi y muestra:
    - status
    - objetivo
    - asignación bloque 1 con edificio
    - asignación bloque 2 con edificio
    """
    carpeta_test = DATA_MODELO_3 / nombre_test
    salas = pd.read_csv(carpeta_test / "salas.csv").set_index("sala_id")

    resultado = resolver_modelo_3(carpeta_test, mostrar_output=False)

    print("Test:", nombre_test)
    print("Status obtenido:", resultado["status"])
    print("Objetivo obtenido:", resultado["objetivo"])

    asignacion_b1 = pd.DataFrame(
        list(resultado["asignacion_b1"].items()),
        columns=["curso_b1", "sala_asignada"]
    )
    asignacion_b1["edificio"] = asignacion_b1["sala_asignada"].map(
        lambda s: salas.loc[s, "edificio"] if s in salas.index else "—"
    )

    asignacion_b2 = pd.DataFrame(
        list(resultado["asignacion_b2"].items()),
        columns=["curso_b2", "sala_asignada"]
    )
    asignacion_b2["edificio"] = asignacion_b2["sala_asignada"].map(
        lambda s: salas.loc[s, "edificio"] if s in salas.index else "—"
    )

    print("\nAsignación bloque 1:")
    display(asignacion_b1)

    print("Asignación bloque 2:")
    display(asignacion_b2)

    return resultado

### M3_T06: 5 cursos en 2 edificios

Este test valida una instancia mediana con 5 cursos, 5 salas distribuidas en 2 edificios. El objetivo es verificar que el modelo prefiere mantener a los estudiantes dentro del mismo edificio cuando el flujo lo permite.

Las salas $r1$ y $r2$ pertenecen al Edificio A; las salas $r3$, $r4$ y $r5$ pertenecen al Edificio B.

Si el flujo de estudiantes está estructurado de modo que los cursos del bloque 2 sean del mismo tamaño que los del bloque 1 y los flujos van mayoritariamente hacia cursos del mismo edificio, la asignación óptima mantendrá a los estudiantes dentro de su edificio. El costo estará dado únicamente por los cambios de sala dentro del mismo edificio, que tienen costo 1.

In [25]:
resultado_t06 = mostrar_resultado_y_asignacion(
    "M3_T06_5_cursos_2_edificios"
)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\yuvi8\\Desktop\\Semestre Actual\\Laboratorio de Modelacion II\\Laboratorio-modelacion-II\\data\\modelo3\\M3_T06_5_cursos_2_edificios\\salas.csv'

### M3_T07: 6 cursos en 3 edificios con estudiantes libres

Este test valida una instancia mediana con 6 cursos en 3 edificios donde todos los cursos del bloque 1 dejan una fracción de estudiantes libres.

Los flujos principales permiten que todos los estudiantes que continúan a un curso del bloque 2 permanezcan en la misma sala. Como los estudiantes libres no generan costo de movilidad, el resultado esperado es costo 0.

In [ ]:
resultado_t07 = mostrar_resultado_y_asignacion(
    "M3_T07_6_cursos_3_edificios_con_libres"
)

### M3_T08: 8 cursos donde los edificios cambian la solución

Este test valida una instancia mediana más grande, con 8 cursos y salas distribuidas en 3 edificios. Es útil para mostrar que el modelo prefiere pagar el costo de un cambio de sala dentro del mismo edificio antes que asignar un curso a un edificio distinto.

El valor de $\gamma$ entre edificios es suficientemente alto como para que el modelo nunca cruce de edificio si hay alguna sala disponible en el mismo edificio. Esto permite verificar que la penalización funciona correctamente.

In [ ]:
resultado_t08 = mostrar_resultado_y_asignacion(
    "M3_T08_8_cursos_edificio_cambia_solucion"
)

### M3_T09: 8 cursos con múltiples óptimos entre edificios

Este test valida una instancia mediana con múltiples soluciones óptimas. Los cursos están organizados en grupos y dentro de cada grupo las salas son del mismo edificio. Los flujos y tamaños están empatados dentro de cada grupo, por lo que Gurobi puede elegir entre varias asignaciones distintas sin cambiar el valor objetivo.

Lo que sí se exige es que cada curso quede asignado a una sala de su grupo correspondiente, y que el costo total sea el esperado.

In [ ]:
resultado_t09 = mostrar_resultado_y_asignacion(
    "M3_T09_8_cursos_multiples_optimos"
)

## 10. Ejecución de todos los tests

Ahora se resuelven todas las carpetas de tests y se comparan los resultados obtenidos contra los resultados esperados definidos en `metadata.json`.

In [ ]:
resultados_tests = []
carpetas_tests = carpetas_tests_pequenos + carpetas_tests_medianos

for carpeta in carpetas_tests:
    with open(carpeta / "metadata.json", "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)

    resultado = resolver_modelo_3(carpeta, mostrar_output=False)

    status_esperado = metadata["expected_status"]
    status_obtenido = resultado["status"]

    objetivo_esperado = metadata["expected_obj"]
    objetivo_obtenido = resultado["objetivo"]

    status_ok = status_esperado == status_obtenido

    if status_esperado == "INFEASIBLE":
        objetivo_ok = True
    else:
        objetivo_ok = abs(objetivo_obtenido - objetivo_esperado) <= 1e-6

    pasa_test = status_ok and objetivo_ok

    resultados_tests.append({
        "test": carpeta.name,
        "status_esperado": status_esperado,
        "status_obtenido": status_obtenido,
        "objetivo_esperado": objetivo_esperado,
        "objetivo_obtenido": objetivo_obtenido,
        "pasa_test": pasa_test
    })

tabla_resultados = pd.DataFrame(resultados_tests)
tabla_resultados

## Interpretación de los resultados

La tabla anterior resume la ejecución de los tests del Modelo 3. Para cada instancia se comparó el status esperado con el status obtenido por Gurobi, y en los casos óptimos también se comparó el valor objetivo esperado con el valor objetivo obtenido.

Si todos los tests pasan correctamente, la columna `pasa_test` toma el valor `True` en todas las filas.

Esto indicaría que la implementación del Modelo 3 se comporta de acuerdo con lo esperado en las instancias sintéticas diseñadas. En particular, se habría verificado que:

- el modelo distingue correctamente los tres tipos de transición (misma sala, mismo edificio, distinto edificio);
- el costo $\gamma$ se aplica correctamente cuando las salas pertenecen a edificios distintos;
- las restricciones de capacidad heredadas del Modelo 2 siguen funcionando correctamente;
- el modelo detecta correctamente una instancia infactible por capacidad;
- los estudiantes libres no generan costo de movilidad;
- el modelo puede manejar casos con múltiples soluciones óptimas;
- las instancias medianas con múltiples edificios son resueltas correctamente.

El caso `M3_T04_infactible_capacidad` aparece con valor objetivo `NaN`, lo cual es correcto, ya que al ser una instancia infactible no existe una solución óptima ni un valor objetivo asociado.

## Caso demostrativo: 16 cursos con múltiples edificios y costos gamma

Después de validar el Modelo 3 con tests pequeños y medianos, se resuelve una instancia más difícil para mostrar las capacidades del modelo en un escenario con múltiples edificios.

Este caso considera 16 cursos en el bloque 1, 16 cursos en el bloque 2 y 16 salas distribuidas en 4 edificios (4 salas por edificio). La penalización $\gamma$ entre edificios distintos es 5.

Los flujos están estructurados de modo que los estudiantes permanecen mayoritariamente dentro de su edificio, pero algunos grupos tienen flujos cruzados entre edificios. Esto permite observar cómo el modelo pondera el costo de quedarse en el mismo edificio versus cruzar a otro.

El resultado esperado surge del análisis de los flujos:

- estudiantes que permanecen en la misma sala: costo 0
- estudiantes que cambian de sala dentro del mismo edificio: costo 1 por estudiante
- estudiantes que cruzan de edificio: costo 5 por estudiante

In [ ]:
from generar_data_modelo_3 import generar_caso_demo_modelo_3

generar_caso_demo_modelo_3()

In [ ]:
from modelo_3_gurobi import mostrar_caso_demo_modelo_3

resultado_demo = mostrar_caso_demo_modelo_3(
    nombre_test="M3_T10_demo_16_cursos_4_edificios"
)

### Interpretación del caso demostrativo

El caso demostrativo permite observar el comportamiento del Modelo 3 en una instancia más exigente. A diferencia de los tests pequeños y medianos, esta instancia tiene más cursos, más salas, múltiples edificios y flujos con distintos tipos de transición. Por lo tanto, se acerca más al tipo de situación que se quiere modelar.

La principal diferencia respecto al caso demostrativo del Modelo 2 es que ahora el costo de moverse entre edificios es considerablemente mayor que moverse dentro del mismo edificio. Esto hace que el modelo prefiera redistribuir los cursos dentro de un edificio antes que enviarlos a otro edificio, incluso si eso implica un costo mayor de cambio de sala.

Las tablas de asignación permiten verificar que el modelo respetó las restricciones de capacidad en ambos bloques y que la distribución de cursos por edificio es consistente con la penalización $\gamma$ definida.

El resumen de movilidad permite distinguir cuántos estudiantes permanecieron en la misma sala, cuántos cambiaron de sala dentro del mismo edificio y cuántos cruzaron de edificio. Esto descompone el valor objetivo en sus tres componentes y permite verificar que la función de costo está bien implementada.

En conclusión, el caso demostrativo funciona como una prueba de capacidad del Modelo 3. Muestra que la formulación puede resolver una instancia con múltiples edificios, flujos mixtos y penalizaciones distintas según el tipo de transición, manteniendo resultados interpretables y verificables.